# BERT Fine-tuning — EMD-based Emotion VAD Prediction

Modernised port of [SungjoonPark/EmotionDetection](https://github.com/SungjoonPark/EmotionDetection).  
Source files referenced:
- `src/models/model.py`   → `PretrainedLMModel`
- `src/models/trainer.py` → `EMDLoss`, `PredcitVADandClassfromLogit`, `Trainer`
- `src/main_ori.py`       → `SingleDatasetTrainer`, training loop

**Key API upgrades from original:**
| Original | Upgraded |
|---|---|
| `from transformers import *` | explicit imports |
| `transformers.AdamW` (deprecated) | `torch.optim.AdamW` |
| `pytorch_pretrained_bert.BertAdam` | removed — use `torch.optim.AdamW` |
| `from_pretrained(..., output_loading_info=True)` | `from_pretrained(...)` (info dropped) |

**Sections**
1. Imports & Config
2. 📥 Data Input *(placeholder)*
3. 🔧 Data Preprocessing *(placeholder)*
4. Dataset & DataLoader
5. Model
6. EMD Loss & VAD Predictor
7. Optimizer & Scheduler
8. Training Loop
9. Evaluation
10. Inference

## 0. Install dependencies

In [ ]:
!pip install transformers datasets torch scipy scikit-learn tqdm -q

## 1. Imports & Config

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
# Explicit imports — replaces `from transformers import *` in original model.py
from transformers import (
    BertTokenizer,
    RobertaTokenizer,
    BertModel,
    RobertaModel,
    BertConfig,
    RobertaConfig,
    BertPreTrainedModel,
    get_linear_schedule_with_warmup,
)
from scipy.stats import pearsonr
from sklearn.metrics import classification_report, jaccard_score
from tqdm.auto import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
# src/main_ori.py → SingleDatasetTrainer._set_model_args()
MODEL_ARCH   = "bert"              # "bert" | "roberta"
MODEL_NAME   = "bert-base-uncased" # swap to "bert-large-cased-whole-word-masking" to match paper
MAX_LEN      = 128
BATCH_SIZE   = 16
EPOCHS       = 3
LR           = 2e-5
LR_UNFREEZE  = 5e-6               # used in two-phase training (Stage 2)
WARMUP_RATIO = 0.1
CLIP_GRAD    = 1.0
UPDATE_FREQ  = 1                  # gradient accumulation steps

# Task: "vad-from-categories" | "category-classification" | "vad-regression"
TASK         = "vad-from-categories"
# Dataset label type: "multi" (multi-label) | "single" (single-label)
LABEL_TYPE   = "multi"

# src/models/trainer.py → Trainer.set_device()
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("Apple Silicon MPS")
else:
    DEVICE = torch.device("cpu")
    print("CPU")

---
## 2. 📥 Data Input

> **PLACEHOLDER** — Load your raw data here.
>
> Expected outputs:
> - `raw_train`, `raw_val`, `raw_test` — raw split data (list / DataFrame / HF DatasetDict)
> - `LABEL_NAMES` — ordered list of emotion category strings, e.g. `["anger", "joy", ...]`
> - `LABEL_VADS`  — dict mapping each label name → `[valence, arousal, dominance]` (NRC scale 0–1)
>
> The SemEval E-c format is shown as the default example (matches `data/SemEval_E-c/`).

In [ ]:
# ── Section 2: Data Input ─────────────────────────────────────────────────────
# Adapted from src/data/loader.py → GOEMOTIONSLoader.__init__() and
#              src/main_ori.py → SingleDatasetTrainer.__init__() lines 56-58

import re
import string
import pandas as pd
from datasets import load_dataset

# ── GoEmotions label list (28 total; neutral is last at index 27) ─────────────
# src/data/loader.py → GOEMOTIONSLoader.__init__()
_ALL_GE_LABELS = [
    "admiration", "amusement",   "anger",      "annoyance",  "approval",
    "caring",     "confusion",   "curiosity",  "desire",     "disappointment",
    "disapproval","disgust",     "embarrassment","excitement","fear",
    "gratitude",  "grief",       "joy",        "love",       "nervousness",
    "optimism",   "pride",       "realization","relief",     "remorse",
    "sadness",    "surprise",    "neutral",                              # index 27
]
NEUTRAL_IDX = 27   # 'neutral' position — excluded from EMD ordering (see CLAUDE.md)

# Stage 1 uses only the 27 non-neutral labels
LABEL_NAMES = _ALL_GE_LABELS[:27]

# ── Load GoEmotions from HuggingFace ──────────────────────────────────────────
# src/data/loader.py → GOEMOTIONSLoader._load_split_files() (HF variant)
print("Loading GoEmotions (simplified) from HuggingFace …")
ds        = load_dataset("go_emotions", "simplified")
raw_train = ds["train"]
raw_val   = ds["validation"]
raw_test  = ds["test"]
print(f"  Train: {len(raw_train):,}  Val: {len(raw_val):,}  Test: {len(raw_test):,}")

# ── NRC VAD Lexicon ───────────────────────────────────────────────────────────
# src/data/loader.py → EmotionDatasetLoader._get_emotion_label_VAD_scores()
# File: data/NRC-VAD-Lexicon-v2.1.txt  (term \t valence \t arousal \t dominance)
NRC_PATH = "data/NRC-VAD-Lexicon-v2.1.txt"

_vad_df = pd.read_csv(NRC_PATH, sep="\t", index_col="term")

LABEL_VADS: dict[str, list[float]] = {}
for label in LABEL_NAMES:
    if label in _vad_df.index:
        row = _vad_df.loc[label]
        LABEL_VADS[label] = [
            round(float(row["valence"]),   3),
            round(float(row["arousal"]),   3),
            round(float(row["dominance"]), 3),
        ]
    else:
        print(f"  WARNING: '{label}' not found in NRC VAD Lexicon — using zeros")
        LABEL_VADS[label] = [0.0, 0.0, 0.0]

# Derived constant (used throughout subsequent cells)
N_LABELS = len(LABEL_NAMES)

print(f"\nLabels ({N_LABELS}): {LABEL_NAMES}")
print("\nSample VAD coords (NRC-VAD-Lexicon-v2.1):")
for lbl in ["joy", "sadness", "anger", "fear", "love"]:
    print(f"  {lbl:15s}: {LABEL_VADS[lbl]}")

---
## 3. 🔧 Data Preprocessing

> **PLACEHOLDER** — Convert raw splits into parallel text / label lists.
>
> Expected outputs:
> - `train_texts`, `val_texts`, `test_texts` — `list[str]`
> - `train_labels`, `val_labels`, `test_labels` — `list[list[int]]` for multi-label  
>   or `list[int]` for single-label classification, `list[list[float]]` for regression

In [ ]:
# ── Section 3: Data Preprocessing ────────────────────────────────────────────
# Adapted from src/data/loader.py → GOEMOTIONSLoader._preprocessing_text()
#              and GOEMOTIONSLoader.load_data()

# ── Text cleaning ─────────────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    """
    Strip surrounding quotes, pad punctuation with spaces, collapse whitespace.
    src/data/loader.py → GOEMOTIONSLoader._preprocessing_text() lines 677-688
    """
    t = str(text).strip('"').strip("'").strip()
    t = re.sub(r"([{}])".format(re.escape(string.punctuation)), r" \1 ", t)
    t = re.sub(r"\s{2,}", " ", t)
    return t.strip()


# ── Label conversion ──────────────────────────────────────────────────────────
# GoEmotions HF 'labels' field = list of active class indices (0-27).
# We produce a 27-dim multi-hot (neutral dropped) for the EMD loss.
# Neutral-only samples → all-zeros vector; the training loop treats these
# as a uniform target distribution (CLAUDE.md: "neutral-only samples use
# a uniform distribution as EMD target").
# src/data/loader.py → GOEMOTIONSLoader._convert_to_one_hot_label() lines 670-675
def to_multihot(label_indices: list) -> list:
    """
    Convert a list of GoEmotions class indices (0-27) into a 27-dim
    multi-hot float vector, ignoring neutral (index 27).
    """
    vec = [0.0] * N_LABELS
    for idx in label_indices:
        if idx != NEUTRAL_IDX:   # skip neutral
            vec[idx] = 1.0
    return vec


# ── Extract splits ────────────────────────────────────────────────────────────
# src/data/loader.py → GOEMOTIONSLoader.load_data() lines 704-726
def extract_split(dataset):
    texts  = [clean_text(ex["text"])    for ex in dataset]
    labels = [to_multihot(ex["labels"]) for ex in dataset]
    return texts, labels

print("Preprocessing splits …")
train_texts, train_labels = extract_split(raw_train)
val_texts,   val_labels   = extract_split(raw_val)
test_texts,  test_labels  = extract_split(raw_test)

# ── Sanity checks ─────────────────────────────────────────────────────────────
print(f"\nTrain: {len(train_texts):,}  Val: {len(val_texts):,}  Test: {len(test_texts):,}")
print(f"Label vector length : {len(train_labels[0])} (should be {N_LABELS})")
print(f"\nSample text  : {train_texts[0]!r}")
print(f"Sample label : {train_labels[0]}")

n_neutral_train = sum(1 for l in train_labels if sum(l) == 0.0)
print(f"\nNeutral-only samples (train): {n_neutral_train:,}/{len(train_labels):,} "
      f"({100*n_neutral_train/len(train_labels):.1f}%)")

---
## 4. Dataset & DataLoader

In [ ]:
# src/data/__init__.py → EmotionDataset.__getitem__()
# Tokenizer selection mirrors main_ori.py → SingleDatasetTrainer.load_tokenizer()

if MODEL_ARCH == "bert":
    tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
else:  # roberta
    tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)


class EmotionDataset(Dataset):
    """
    Wraps texts + labels for DataLoader consumption.
    For LABEL_TYPE='multi'  labels are float tensors (BCEWithLogitsLoss compatible).
    For LABEL_TYPE='single' labels are long tensors  (CrossEntropyLoss compatible).
    For vad-regression      labels are float tensors shaped (N, 3).
    """

    def __init__(self, texts, labels):
        self.texts  = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        raw_label = self.labels[idx]
        if LABEL_TYPE == "single":
            label_tensor = torch.tensor(raw_label, dtype=torch.long)
        else:  # multi or regression
            label_tensor = torch.tensor(raw_label, dtype=torch.float)

        return (
            enc["input_ids"].squeeze(0),       # (MAX_LEN,)
            enc["attention_mask"].squeeze(0),  # (MAX_LEN,)
            label_tensor,                      # (N_LABELS,) or scalar
        )


train_ds = EmotionDataset(train_texts, train_labels)
val_ds   = EmotionDataset(val_texts,   val_labels)
test_ds  = EmotionDataset(test_texts,  test_labels) if test_texts else None

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = (DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
                if test_ds else None)

print(f"Train batches: {len(train_loader)}  Val batches: {len(val_loader)}")

---
## 5. Model

Ported from `src/models/model.py → PretrainedLMModel`.

**Changes from original:**
- Removed `from transformers import *` → explicit imports
- Removed `output_loading_info=True` (deprecated kwarg, info was only printed)
- Kept `return_dict=False` for the `(hidden_states, pooled_output)` unpack pattern
- `vad-regression` frozen-head initialisation kept intact

In [ ]:
# src/models/model.py → PretrainedLMModel
class PretrainedLMModel(BertPreTrainedModel):
    """
    BERT (or RoBERTa) backbone with a task-specific head.

    Tasks
    -----
    vad-from-categories : head → (B, N_LABELS*3); split into V/A/D logits
    category-classification : head → (B, N_LABELS)
    vad-regression      : head → (B, N_LABELS*3) or (B, 3) depending on checkpoint
    """

    def __init__(self, config):
        super().__init__(config)
        self.config = config
        args = config.args

        # ── Backbone ─────────────────────────────────────────────────────────
        # Original used cache_dir; removed here for simplicity
        if args.model == "bert":
            self.pre_trained_lm = BertModel.from_pretrained(MODEL_NAME)
        else:  # roberta
            self.pre_trained_lm = RobertaModel.from_pretrained(MODEL_NAME)

        self.dropout = nn.Dropout(config.hidden_dropout_prob)

        # ── Head ─────────────────────────────────────────────────────────────
        if args.task == "vad-regression":
            n_out = N_LABELS * 3 if args.load_checkpoint else 3
            self.head = nn.Linear(config.hidden_size, n_out)

            # Frozen VAD-initialised sub-heads (used when loading a checkpoint)
            # src/models/model.py lines 64-85
            if args.load_checkpoint:
                v_scores = [LABEL_VADS[k][0] for k in LABEL_NAMES]
                a_scores = [LABEL_VADS[k][1] for k in LABEL_NAMES]
                d_scores = [LABEL_VADS[k][2] for k in LABEL_NAMES]
                v_vals = torch.tensor(sorted(v_scores), dtype=torch.float).to(DEVICE)
                a_vals = torch.tensor(sorted(a_scores), dtype=torch.float).to(DEVICE)
                d_vals = torch.tensor(sorted(d_scores), dtype=torch.float).to(DEVICE)
                self.v_head = nn.Linear(N_LABELS, 1, bias=False)
                self.a_head = nn.Linear(N_LABELS, 1, bias=False)
                self.d_head = nn.Linear(N_LABELS, 1, bias=False)
                self.v_head.weight = nn.Parameter(v_vals.unsqueeze(0))
                self.a_head.weight = nn.Parameter(a_vals.unsqueeze(0))
                self.d_head.weight = nn.Parameter(d_vals.unsqueeze(0))

        elif args.task == "vad-from-categories":
            self.head = nn.Linear(config.hidden_size, N_LABELS * 3)

        else:  # category-classification
            self.head = nn.Linear(config.hidden_size, N_LABELS)

        self.post_init()

    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        # return_dict=False → tuple: (last_hidden_state, pooler_output)
        # src/models/model.py lines 109-129
        lm_out = self.pre_trained_lm(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            return_dict=False,
        )
        _, pooled = lm_out[0], lm_out[1]   # (B, seq, H), (B, H)
        pooled  = self.dropout(pooled)
        logits  = self.head(pooled)         # (B, head_size)

        # VAD sub-head aggregation (vad-regression with checkpoint)
        # src/models/model.py lines 132-141
        if (hasattr(self.config.args, "load_checkpoint")
                and self.config.args.task == "vad-regression"
                and self.config.args.load_checkpoint):
            v_logit, a_logit, d_logit = torch.split(logits, N_LABELS, dim=1)
            logits = torch.cat([
                self.v_head(torch.sigmoid(v_logit)),
                self.a_head(torch.sigmoid(a_logit)),
                self.d_head(torch.sigmoid(d_logit)),
            ], dim=1)

        return logits


# ── Build model ───────────────────────────────────────────────────────────────
import argparse

_args = argparse.Namespace(
    model=MODEL_ARCH,
    task=TASK,
    load_checkpoint=False,   # set True for Stage-2 / vad-regression with prior checkpoint
)

if MODEL_ARCH == "bert":
    _config = BertConfig.from_pretrained(MODEL_NAME)
else:
    _config = RobertaConfig.from_pretrained(MODEL_NAME)

_config.args = _args
model = PretrainedLMModel(_config).to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total:,}")
print(f"Trainable params: {trainable:,}")

---
## 6. EMD Loss & VAD Predictor

Ported from `src/models/trainer.py → EMDLoss` and `PredcitVADandClassfromLogit`.

**EMD Loss** combines two terms per VAD dimension:
- **Inter-EMD**: distance-weighted squared CDF difference (accounts for actual VAD gap between adjacent emotions)
- **Intra-EMD**: mean squared difference without distance weighting

**Changes from original:**
- Kept exact arithmetic; only removed `self.args` dependency → accepts `LABEL_NAMES` / `LABEL_VADS` directly
- `BertAdam` (pytorch_pretrained_bert) removed entirely — see Section 7

In [ ]:
# src/models/trainer.py → EMDLoss (lines 23-150)

class EMDLoss(nn.Module):
    """
    Earth Mover's Distance loss over V, A, D axes independently.
    Each axis sorts emotion categories by their VAD score and computes
    inter-EMD (distance-weighted CDF diff) + intra-EMD (MSE of probs).
    """

    def __init__(self, label_names, label_vads, label_type="multi"):
        super().__init__()
        assert label_type in ("single", "multi")
        self.label_names = label_names
        self.label_vads  = label_vads
        self.label_type  = label_type
        self.n_labels    = len(label_names)
        self.eps         = 1e-5

        if label_type == "single":
            self.activation = nn.Softmax(dim=1)
        else:
            self.activation = nn.Sigmoid()

        self._sort_labels()

    def _sort_labels(self):
        # src/models/trainer.py → EMDLoss._sort_labels() lines 55-69
        for dim_idx, dim_name in enumerate(["v", "a", "d"]):
            scores = [self.label_vads[k][dim_idx] for k in self.label_names]
            sorted_idxs   = torch.tensor(np.argsort(scores).tolist())
            sorted_values = torch.tensor(np.sort(scores).tolist(), dtype=torch.float)
            setattr(self, f"{dim_name}_sorted_idxs",   sorted_idxs)
            setattr(self, f"{dim_name}_sorted_values", sorted_values)

    def _to_device(self, t):
        return t.to(next(self.parameters()).device) if len(list(self.parameters())) else t.to(DEVICE)

    def _sort_by_dim(self, labels, sorted_idxs):
        # src/models/trainer.py → EMDLoss._sort_labels_by_vad_coordinates() lines 71-75
        return torch.index_select(labels, 1, sorted_idxs.to(labels.device))

    def _distance_vector(self, sorted_values):
        # src/models/trainer.py → EMDLoss._set_vad_distance_matrix() lines 77-96
        # Gap between adjacent sorted VAD values; last element = 0
        d = torch.roll(sorted_values, -1, 0) - sorted_values
        for i in range(len(d) - 1):
            if d[i] == 0:
                d[i] = d[i + 1]
        d[-1] = 0.0
        return d.to(DEVICE)

    def _intra_emd(self, pred_probs, label_probs):
        # src/models/trainer.py → EMDLoss._intra_EMD_loss() lines 98-101
        return torch.div(
            torch.sum(torch.square(pred_probs - label_probs), dim=1),
            self.n_labels
        )

    def _inter_emd(self, pred_probs, label_probs, distance):
        # src/models/trainer.py → EMDLoss._inter_EMD_loss() lines 104-112
        norm_pred  = pred_probs  / (pred_probs.sum(dim=1, keepdim=True)  + self.eps)
        norm_label = label_probs / (label_probs.sum(dim=1, keepdim=True) + self.eps)
        cdf_diff_sq = torch.square(
            torch.cumsum(norm_pred, dim=1) - torch.cumsum(norm_label, dim=1)
        )
        return torch.matmul(distance, cdf_diff_sq.T)  # (B,)

    def forward(self, logits, labels):
        """
        logits : (B, N_LABELS * 3) — concatenated V/A/D logits
        labels : (B, N_LABELS)     — multi-hot or one-hot
        src/models/trainer.py → EMDLoss.forward() lines 115-150
        """
        if self.label_type == "single":
            one_hot = torch.eye(self.n_labels, device=labels.device)
            labels  = one_hot[labels]

        # Split logits into per-dimension blocks (each sorted by that VAD dim)
        v_logit, a_logit, d_logit = torch.split(logits, self.n_labels, dim=1)

        losses = []
        for logit, dim_name in zip([v_logit, a_logit, d_logit], ["v", "a", "d"]):
            sorted_idxs   = getattr(self, f"{dim_name}_sorted_idxs")
            sorted_values = getattr(self, f"{dim_name}_sorted_values")
            sorted_labels = self._sort_by_dim(labels, sorted_idxs)  # reorder label cols
            distance      = self._distance_vector(sorted_values)

            pred_probs    = self.activation(logit)
            inter         = self._inter_emd(pred_probs, sorted_labels, distance)
            intra         = self._intra_emd(pred_probs, sorted_labels)
            losses.append(inter + intra)

        loss = torch.mean(torch.stack(losses, dim=1), dim=1)  # mean over V/A/D
        return loss


# src/models/trainer.py → PredcitVADandClassfromLogit (lines 155-235)
class PredictVADandClass(nn.Module):
    """
    Converts raw logits from the model head into VAD scores or category predictions.
    VAD  → expected value under the predicted distribution (E[vad] = probs · vad_values)
    Cat  → argmax / sigmoid-threshold over the combined V+A+D logits
    """

    def __init__(self, label_names, label_vads, label_type="multi"):
        super().__init__()
        assert label_type in ("single", "multi")
        self.n_labels   = len(label_names)
        self.label_type = label_type

        if label_type == "single":
            self.activation = nn.Softmax(dim=1)
        else:
            self.activation = nn.Sigmoid()

        for dim_idx, dim_name in enumerate(["v", "a", "d"]):
            scores        = [label_vads[k][dim_idx] for k in label_names]
            sorted_idxs   = torch.tensor(np.argsort(scores).tolist())
            recover_idxs  = torch.argsort(sorted_idxs)
            sorted_values = torch.tensor(np.sort(scores).tolist(), dtype=torch.float)
            self.register_buffer(f"{dim_name}_sorted_idxs",   sorted_idxs)
            self.register_buffer(f"{dim_name}_recover_idxs",  recover_idxs)
            self.register_buffer(f"{dim_name}_sorted_values", sorted_values)

    def forward(self, logits, predict="vad"):
        """
        predict: "vad" → (B, 3) continuous VAD scores
                 "cat" → (B, N) binary predictions or (B,) class indices
        src/models/trainer.py → PredcitVADandClassfromLogit.forward() lines 202-235
        """
        v_logit, a_logit, d_logit = torch.split(logits, self.n_labels, dim=1)
        v_probs = self.activation(v_logit)
        a_probs = self.activation(a_logit)
        d_probs = self.activation(d_logit)

        if predict == "vad":
            e_v = (v_probs * self.v_sorted_values).sum(dim=1)
            e_a = (a_probs * self.a_sorted_values).sum(dim=1)
            e_d = (d_probs * self.d_sorted_values).sum(dim=1)
            return torch.stack([e_v, e_a, e_d], dim=1)   # (B, 3)

        else:  # cat
            v_orig = torch.index_select(v_logit, 1, self.v_recover_idxs)
            a_orig = torch.index_select(a_logit, 1, self.a_recover_idxs)
            d_orig = torch.index_select(d_logit, 1, self.d_recover_idxs)
            combined = v_orig + a_orig + d_orig
            if self.label_type == "multi":
                log_p = (combined
                         - torch.log(torch.exp(v_orig) + 1)
                         - torch.log(torch.exp(a_orig) + 1)
                         - torch.log(torch.exp(d_orig) + 1))
                return (torch.exp(log_p).pow(1 / 3) >= 0.5).float().squeeze()
            else:
                return combined.argmax(dim=1)


# Instantiate loss and predictor
criterion     = EMDLoss(LABEL_NAMES, LABEL_VADS, label_type=LABEL_TYPE)
vad_predictor = PredictVADandClass(LABEL_NAMES, LABEL_VADS, label_type=LABEL_TYPE).to(DEVICE)
print("EMDLoss and PredictVADandClass ready.")

---
## 7. Optimizer & Scheduler

Ported from `src/models/trainer.py → Trainer.set_optimizer()`.

**Change:** `transformers.AdamW` (deprecated since transformers 4.x) → `torch.optim.AdamW`.  
The legacy `BertAdam` from `pytorch_pretrained_bert` is removed entirely.

In [ ]:
# src/models/trainer.py → Trainer.set_optimizer() lines 356-369
# UPGRADE: transformers.AdamW → torch.optim.AdamW (transformers version deprecated)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    betas=(0.9, 0.98),
    eps=1e-6,
    weight_decay=0.01,
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"Total steps  : {total_steps}")
print(f"Warmup steps : {warmup_steps}")

---
## 8. Training Loop

Adapted from `src/main_ori.py → SingleDatasetTrainer.train()` and  
`src/models/trainer.py → Trainer.backward_step()`.

In [ ]:
# src/main_ori.py → SingleDatasetTrainer.train() lines 319-443
# src/models/trainer.py → Trainer.backward_step() lines 384-399

def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss  = 0.0
    n_updates   = 0
    accum_loss  = torch.tensor(0.0).to(DEVICE)

    pbar = tqdm(loader, desc="  train", leave=False)
    for it, batch in enumerate(pbar):
        input_ids      = batch[0].to(DEVICE)
        attention_mask = batch[1].to(DEVICE)
        labels         = batch[2].to(DEVICE)

        logits = model(input_ids, attention_mask=attention_mask)

        # EMD loss returns per-sample losses; take mean
        loss       = torch.mean(criterion(logits, labels))
        accum_loss = accum_loss + loss
        loss.backward()

        # Gradient accumulation (src/models/trainer.py lines 388-398)
        if (it + 1) % UPDATE_FREQ == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            n_updates += 1

        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(loader)


# ── Main loop ─────────────────────────────────────────────────────────────────
best_val_loss = float("inf")
optimizer.zero_grad()

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    train_loss = train_epoch(model, train_loader, optimizer, scheduler)
    print(f"  Train loss: {train_loss:.4f}")

    # Evaluation block (detailed in Section 9)
    # val_loss, val_metrics = evaluate(model, val_loader)
    # if val_loss < best_val_loss:
    #     best_val_loss = val_loss
    #     torch.save(model.state_dict(), "demo_best_model.pt")
    #     print("  ✓ Saved best model")

---
## 9. Evaluation

Ported from `src/models/trainer.py → Trainer.predict()` and `Trainer.compute_eval_metric()`.

In [ ]:
# src/models/trainer.py → Trainer.predict() lines 464-524
# src/models/trainer.py → Trainer.compute_eval_metric() lines 423-461

@torch.no_grad()
def evaluate(model, loader, prediction_type="cat"):
    """
    prediction_type: "cat" → classification metrics (F1, Jaccard)
                     "vad" → Pearson r per dimension
    """
    model.eval()
    total_loss   = 0.0
    all_preds    = []
    all_labels   = []

    for batch in tqdm(loader, desc="  eval", leave=False):
        input_ids      = batch[0].to(DEVICE)
        attention_mask = batch[1].to(DEVICE)
        labels         = batch[2].to(DEVICE)

        logits = model(input_ids, attention_mask=attention_mask)
        loss   = torch.mean(criterion(logits, labels))
        total_loss += loss.item()

        # Convert logits → predictions
        # src/models/trainer.py → Trainer.predict() lines 491-516
        preds = vad_predictor(logits, predict=prediction_type)

        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

    preds_all  = torch.cat(all_preds,  dim=0).numpy()
    labels_all = torch.cat(all_labels, dim=0).numpy()
    avg_loss   = total_loss / len(loader)
    metrics    = {}

    if prediction_type == "vad":
        # src/models/trainer.py → Trainer._compute_vad_eval_metrics() lines 402-405
        for i, name in enumerate(["v_cor", "a_cor", "d_cor"]):
            metrics[name] = pearsonr(preds_all[:, i], labels_all[:, i])
        print(f"  Val loss : {avg_loss:.4f}")
        for k, (r, p) in metrics.items():
            print(f"  {k}  r={r:.4f}  p={p:.4f}")

    else:  # cat
        # src/models/trainer.py → Trainer._compute_classification_eval_metrics() lines 408-420
        report = classification_report(
            labels_all, preds_all, digits=4, zero_division=0, output_dict=True
        )
        jaccard_micro  = jaccard_score(labels_all, preds_all, average="micro")
        jaccard_macro  = jaccard_score(labels_all, preds_all, average="macro")
        jaccard_sample = jaccard_score(labels_all, preds_all, average="samples")
        metrics["classification"] = report
        metrics["jaccard"] = {"micro": jaccard_micro, "macro": jaccard_macro, "samples": jaccard_sample}
        print(f"  Val loss    : {avg_loss:.4f}")
        print(f"  Micro-F1    : {report['micro avg']['f1-score']:.4f}")
        print(f"  Macro-F1    : {report['macro avg']['f1-score']:.4f}")
        print(f"  Jaccard-micro : {jaccard_micro:.4f}")

    return avg_loss, metrics


# Example call (requires trained model and filled data loaders)
# val_loss, val_metrics = evaluate(model, val_loader, prediction_type="cat")

---
## 10. Inference

In [ ]:
# src/models/trainer.py → Trainer.predict() single-sample adaptation

model.load_state_dict(torch.load("demo_best_model.pt", map_location=DEVICE))
model.eval()


@torch.no_grad()
def predict(text: str, top_k: int = 3):
    enc = tokenizer(
        text,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).to(DEVICE)

    logits = model(enc["input_ids"], attention_mask=enc["attention_mask"])

    # VAD scores
    vad   = vad_predictor(logits, predict="vad").squeeze(0).cpu().numpy()
    # Category predictions
    cats  = vad_predictor(logits, predict="cat").squeeze(0).cpu().numpy()

    print(f"Text: {text!r}")
    print(f"\nPredicted categories:")
    active = [LABEL_NAMES[i] for i, v in enumerate(cats) if v > 0.5]
    print(f"  {active if active else '(none above threshold)'}")
    print(f"\nPredicted VAD (zero-shot expected value):")
    print(f"  Valence   = {vad[0]:.4f}")
    print(f"  Arousal   = {vad[1]:.4f}")
    print(f"  Dominance = {vad[2]:.4f}")


# ── Demo calls — replace with real examples from your dataset ─────────────────
predict("I can't believe we won. This is incredible!")
predict("I'm exhausted and nothing seems to matter anymore.")